# Lesson 03 — RANSAC: Rejecting Outliers

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img1 = cv2.imread('sample.jpg')
img2 = img1.copy()
M_sim = cv2.getRotationMatrix2D((img1.shape[1]//2,img1.shape[0]//2),15,0.9)
img2  = cv2.warpAffine(img2, M_sim, (img2.shape[1],img2.shape[0]))

sift = cv2.SIFT_create(nfeatures=400)
kp1,d1 = sift.detectAndCompute(cv2.cvtColor(img1,cv2.COLOR_BGR2GRAY),None)
kp2,d2 = sift.detectAndCompute(cv2.cvtColor(img2,cv2.COLOR_BGR2GRAY),None)
bf     = cv2.BFMatcher()
good   = [m for m,n in bf.knnMatch(d1,d2,k=2) if m.distance < 0.75*n.distance]

src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)

# RANSAC threshold comparison
for thresh in [1.0, 3.0, 5.0, 10.0]:
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, thresh)
    print(f"RANSAC thresh={thresh:4.1f}: inliers={mask.sum():3d}/{len(good)}")

H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
inlier_matches  = [good[i] for i in range(len(good)) if mask[i]]
outlier_matches = [good[i] for i in range(len(good)) if not mask[i]]

vis_in  = cv2.drawMatches(img1,kp1,img2,kp2,inlier_matches[:40], None,matchColor=(0,255,0),flags=2)
vis_out = cv2.drawMatches(img1,kp1,img2,kp2,outlier_matches[:40],None,matchColor=(0,0,255),flags=2)

fig,axes = plt.subplots(1,2,figsize=(18,5))
axes[0].imshow(cv2.cvtColor(vis_in, cv2.COLOR_BGR2RGB)); axes[0].set_title(f'INLIERS ({len(inlier_matches)}) — consistent')
axes[1].imshow(cv2.cvtColor(vis_out,cv2.COLOR_BGR2RGB)); axes[1].set_title(f'OUTLIERS ({len(outlier_matches)}) — wrong')
plt.show()

## Key Takeaway
RANSAC randomly samples 4 point pairs, computes H, counts how many other points agree.
Repeats many times. Returns H that the most points agree with.
reprojThresh=5 pixels is a universal safe default.